# bencheetah — Tutorial

> Simple, zero-dependency benchmarking utilities for Python functions.

[![PyPI](https://img.shields.io/pypi/v/bencheetah.svg)](https://pypi.org/project/bencheetah/)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JobAlcantara/bencheetah/blob/main/notebooks/tutorial.ipynb)

---

This notebook walks you through all four public functions of **bencheetah**:

| Function | Purpose |
|---|---|
| `benchmark()` | Time a single function across multiple runs |
| `compare()` | Race several functions on the same input |
| `format_results()` | Pretty-print results as a terminal table |
| `scale_benchmark()` | Sweep input sizes and collect timing data |
| `plot_scaling()` | Plot mean time vs input size with matplotlib |

## 0. Installation

In [1]:
%pip install "bencheetah[plot]" -q

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement bencheetah[plot] (from versions: none)
ERROR: No matching distribution found for bencheetah[plot]


In [2]:
import bencheetah
print('bencheetah version:', bencheetah.__version__)

ModuleNotFoundError: No module named 'bencheetah'

## 1. Import

In [ ]:
from bencheetah import benchmark, compare, format_results, scale_benchmark, plot_scaling

---
## 2. `benchmark()` — Test a single function

`benchmark(func, *args, repeats=5, warmup=1, **kwargs)`

- Runs `func` once as a **warm-up** (eliminates cold-start noise).
- Then runs it `repeats` times and collects mean, min, max, stdev.

In [ ]:
result = benchmark(sum, range(1_000_000), repeats=10)
print(format_results(result))

In [ ]:
def bubble_sort(lst):
    lst = lst[:]
    n = len(lst)
    for i in range(n):
        for j in range(n - i - 1):
            if lst[j] > lst[j + 1]:
                lst[j], lst[j + 1] = lst[j + 1], lst[j]
    return lst

data = list(range(300, 0, -1))
result = benchmark(bubble_sort, data, repeats=5, warmup=1)
print(format_results(result))

---
## 3. `compare()` — Race multiple implementations

`compare(funcs, args=None, kwargs=None, repeats=5, warmup=1)`

Pass a `{"label": callable}` dict and shared inputs via `args`.

In [ ]:
def loop_sum(n): return sum(range(n))
def gauss_sum(n): return n * (n - 1) // 2

out = compare(
    {"loop sum": loop_sum, "gauss formula": gauss_sum},
    args=(1_000_000,),
    repeats=8,
)
print(format_results(out))

In [ ]:
import random
sample = random.sample(range(100_000), 2_000)

def use_sorted(lst): return sorted(lst)
def use_list_sort(lst):
    lst = lst[:]; lst.sort(); return lst

out = compare(
    {"sorted()": use_sorted, "list.sort()": use_list_sort, "bubble sort": bubble_sort},
    args=(sample,),
    repeats=6, warmup=2,
)
print(format_results(out))

---
## 4. `scale_benchmark()` — Execution time vs input size

```python
scale_benchmark(funcs, input_gen, sizes, repeats=5, warmup=1)
```

- `funcs` — a single callable **or** a `{"label": callable}` dict.
- `input_gen(n)` — called once per size; return a `tuple` for multiple args.
- `sizes` — the list of *n* values to sweep.

Returns a dict with `sizes`, `means`, `stdevs`, `mins`, `maxs` per function.

In [ ]:
def insertion_sort(lst):
    lst = lst[:]
    for i in range(1, len(lst)):
        key = lst[i]
        j = i - 1
        while j >= 0 and lst[j] > key:
            lst[j + 1] = lst[j]
            j -= 1
        lst[j + 1] = key
    return lst

SIZES = [100, 500, 1_000, 2_000, 5_000]

scale_data = scale_benchmark(
    funcs={
        "sorted() built-in": sorted,
        "insertion sort":    insertion_sort,
        "bubble sort":       bubble_sort,
    },
    input_gen=lambda n: list(range(n, 0, -1)),  # worst-case: reversed input
    sizes=SIZES,
    repeats=5,
    warmup=1,
)

# Inspect the raw data
for name, entry in scale_data.items():
    print(f'{name}')
    for n, m in zip(entry['sizes'], entry['means']):
        print(f'  n={n:>5,}  {m*1000:.3f} ms')
    print()

---
## 5. `plot_scaling()` — Chart the results

```python
plot_scaling(scale_data, *, title, time_unit, show_errorbars,
             show_minmax_band, logscale, figsize, save_path, show)
```

X axis = input size, Y axis = mean execution time.  
Error bars show ±1 standard deviation.

In [ ]:
fig, ax = plot_scaling(
    scale_data,
    title="Sort algorithm scaling — worst-case (reversed input)",
    show_errorbars=True,
)

### With min–max shading

In [ ]:
fig, ax = plot_scaling(
    scale_data,
    title="Sort algorithm scaling — with min/max band",
    show_errorbars=False,
    show_minmax_band=True,
)

### Log scale — useful when algorithms differ by orders of magnitude

In [ ]:
fig, ax = plot_scaling(
    scale_data,
    title="Sort algorithm scaling — log scale",
    logscale=True,
)

### Customise the axes after the fact

`plot_scaling` returns `(fig, ax)` so you can add annotations or extra formatting.

In [ ]:
fig, ax = plot_scaling(
    scale_data,
    title="Customised chart",
    show=False,  # don't show yet
)

ax.annotate(
    "O(n²) territory",
    xy=(5000, ax.get_lines()[-1].get_ydata()[-1]),
    xytext=(3500, ax.get_ylim()[1] * 0.8),
    arrowprops=dict(arrowstyle="->", color="gray"),
    fontsize=9, color="gray",
)

import matplotlib.pyplot as plt
plt.show()

### Save to file

In [ ]:
fig, ax = plot_scaling(
    scale_data,
    title="Sort algorithm scaling",
    save_path="scaling_sort.png",
    show=True,
)
print('Saved to scaling_sort.png')

---
## 6. Working with raw scale data

The dict from `scale_benchmark()` is plain Python — feed it into pandas, plot with seaborn, or process however you like.

In [ ]:
import pandas as pd

rows = []
for name, entry in scale_data.items():
    for n, mean, stdev in zip(entry['sizes'], entry['means'], entry['stdevs']):
        rows.append({'algorithm': name, 'n': n,
                     'mean_ms': mean * 1000, 'stdev_ms': stdev * 1000})

df = pd.DataFrame(rows)
df

---
## 7. Summary

```python
from bencheetah import benchmark, compare, format_results, scale_benchmark, plot_scaling

# Single function
print(format_results(benchmark(my_func, *args, repeats=10)))

# Compare at one input size
print(format_results(compare({"A": fa, "B": fb}, args=(...,), repeats=10)))

# Scaling experiment + chart
data = scale_benchmark({"A": fa, "B": fb}, input_gen=lambda n: ..., sizes=[...])
plot_scaling(data, title="My experiment")
```

[PyPI](https://pypi.org/project/bencheetah/) &nbsp;|&nbsp; [GitHub](https://github.com/JobAlcantara/bencheetah)